In [0]:
%sql
CREATE CATALOG IF NOT EXISTS online;
 
CREATE SCHEMA IF NOT EXISTS online.retail_analytics;

In [0]:
%sql
CREATE TABLE online.retail_analytics.customers (
    customer_id INT,
    first_name STRING,
    last_name STRING,
    city STRING,
    signup_date STRING
);
 
CREATE TABLE online.retail_analytics.products (
    product_id INT,
    product_name STRING,
    category STRING
);
 
CREATE TABLE online.retail_analytics.orders (
    order_id INT,
    customer_id INT,
    product_id INT,
    quantity INT,
    price DOUBLE,
    order_date STRING,
    region STRING
);

In [0]:
customers = spark.read.csv("/Volumes/online/retail_analytics/orders_product_customers/customers.csv", header=True, inferSchema=True)
orders = spark.read.csv("/Volumes/online/retail_analytics/orders_product_customers/orders (3).csv", header=True, inferSchema=True)
products = spark.read.csv("/Volumes/online/retail_analytics/orders_product_customers/products.csv", header=True, inferSchema=True)

### Understant lazy Execution and how builds sparks Execution plan

In [0]:
df_filtered = orders.filter("price > 500")

df_final = df_filtered.withColumn(
    "total_amount",
    df_filtered["quantity"] * df_filtered["price"]
)

# Action
df_final.show()

In [0]:
from pyspark.sql.functions import sum, col

 
 
df_group = orders.withColumn(

    "total", col("quantity") * col("price")

).groupBy("region").agg(

    sum("total").alias("total_sales")

)
 
df_group.show()

In [0]:
df_group.filter("total_sales > 5000").show()

Ranking (Top orders)


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
 
 
 
window_spec = Window.partitionBy("region").orderBy(col("price").desc())
 
df_ranked = orders.withColumn("rn", row_number().over(window_spec))
 
df_top3 = df_ranked.filter("rn <= 3")
 
df_top3.show()

In [0]:
df_join = orders.join(

    customers,

    orders["customer_id"] == customers["customer_id"],

    "inner"

).join(

    products,

    orders["product_id"] == products["product_id"],

    "inner"

)
 
df_join.show()
 

In [0]:
display(df_join)

In [0]:
from pyspark.sql.functions import to_date, current_date, datediff
 
 
 
df = orders.withColumn(

    "order_date",

    to_date("order_date", "yyyy-MM-dd")

)
 
df = df.withColumn("today", current_date())
 
df = df.withColumn(

    "days_since_order",

    datediff("today", "order_date")

)
 
df.show()
 

In [0]:
from pyspark.sql.functions import to_date, current_date, datediff,month,dayofmonth
 
df_date = orders.withColumn(
    "order_date",
    to_date("order_date", "yyyy-MM-dd")
)
 
df_date_month = df_date.withColumn("order_month",month("order_date")).withColumn("order_day",dayofmonth("order_date"))

display(df_date_month)

In [0]:
from pyspark.sql.functions import to_date, current_date, datediff,month,dayofmonth

df_date_month=df.withcolumn("order_month",month("order_date")).withcolumn("oerder_day",dayofmonth("order_date"))

In [0]:
df_order_over_month = df_date_month.groupBy("order_day").agg({"order_id": "count"}).show()
display(df_order_over_month)